# 02. Cleaning & Preprocessing

## Digital Burnout International Dataset

- Xử lý các vấn đề dữ liệu được phát hiện trong PreEDA.
- Chuẩn hóa dữ liệu phục vụ phân tích tiếp theo.
- Tạo bộ dữ liệu sạch cho EDA và Feature Validation.

## Import thư viện

In [1]:
# Import libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Raw Dataset

Notebook này sử dụng dữ liệu thô từ thư mục `data/raw`.

Dữ liệu đầu vào là kết quả trước khi cleaning, chưa được encoding, scaling hoặc feature engineering.

In [9]:
pd.set_option("display.max_columns", None)

file_path = "../../data/raw/digital_burnout_productivity_dataset_5M.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")

Dataset loaded successfully.
Shape: (5000000, 34)


## 2. Create Working Copy

Tạo bản sao của dataset để xử lý.

Việc này giúp giữ nguyên dữ liệu gốc và đảm bảo các bước cleaning có thể được kiểm soát rõ ràng.

In [4]:
# Create working copy

df_cleaned = df.copy()

print("Working copy created.")
print(f"Initial shape: {df_cleaned.shape}")

Working copy created.
Initial shape: (5000000, 34)


## 3. Cleaning Decision Log

Phần này ghi lại các quyết định tiền xử lý được thực hiện trong notebook nhằm đảm bảo tính minh bạch của nghiên cứu.

In [10]:
cleaning_log = []

## 4. Remove Duplicate Records

Các bản ghi trùng lặp hoàn toàn sẽ được loại bỏ nhằm tránh ảnh hưởng đến các phân tích tiếp theo.

In [11]:
duplicates_before = df_cleaned.duplicated().sum()

df_cleaned = df_cleaned.drop_duplicates()

duplicates_removed = duplicates_before

cleaning_log.append({
    "step": "Remove Duplicates",
    "affected_records": duplicates_removed,
    "decision": "Dropped duplicated rows"
})

print(f"Duplicates removed: {duplicates_removed}")

Duplicates removed: 0


## 5. Handle Missing Values

Chiến lược xử lý:

- Numerical variables → Median
- Categorical variables → Mode

Mục tiêu là giữ lại tối đa dữ liệu phục vụ các bước phân tích tiếp theo.

In [14]:
numerical_columns = df_cleaned.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_columns = df_cleaned.select_dtypes(
    include=["object", "category", "bool"]
).columns

for col in numerical_columns:

    if df_cleaned[col].isnull().sum() > 0:

        median_value = df_cleaned[col].median()

        df_cleaned[col] = df_cleaned[col].fillna(
            median_value
        )
        
for col in categorical_columns:

    if df_cleaned[col].isnull().sum() > 0:

        mode_value = df_cleaned[col].mode()

        fill_value = (
            mode_value[0]
            if len(mode_value) > 0
            else "Unknown"
        )

        df_cleaned[col] = df_cleaned[col].fillna(
            fill_value
        )

In [15]:
cleaning_log.append({
    "step": "Handle Missing Values",
    "decision": "Median for numerical, mode for categorical"
})

## 6. Data Type Validation

Chuẩn hóa kiểu dữ liệu theo thiết kế nghiên cứu nhằm đảm bảo tính nhất quán cho các bước tiếp theo.

In [16]:
expected_numerical_columns = [
    "burnout_risk",
    "emotional_exhaustion",
    "stress_level",
    "mental_fatigue",
    "daily_screen_time",
    "social_media_hours",
    "doomscrolling_duration",
    "late_night_device_usage",
    "notification_count",
    "smartphone_unlocks",
    "app_switch_frequency",
    "concentration_score",
    "distraction_frequency",
    "focus_sessions",
    "deep_work_hours",
    "task_completion_rate",
    "motivation_level",
    "sleep_hours",
    "sleep_quality",
    "productivity_score"
]

for col in expected_numerical_columns:

    if col in df_cleaned.columns:

        df_cleaned[col] = pd.to_numeric(
            df_cleaned[col],
            errors="coerce"
        )

In [17]:
for col in df_cleaned.columns:

    if df_cleaned[col].isnull().sum() > 0:

        if col in expected_numerical_columns:

            fill_value = df_cleaned[col].median()

        else:

            mode_value = df_cleaned[col].mode()

            fill_value = (
                mode_value[0]
                if len(mode_value) > 0
                else "Unknown"
            )

        df_cleaned[col] = df_cleaned[col].fillna(
            fill_value
        )

In [18]:
cleaning_log.append({
    "step": "Data Type Validation",
    "decision": "Converted expected research variables"
})

## 7. Research Variable Inventory

Phần này chỉ kiểm tra sự hiện diện của các biến nghiên cứu liên quan Digital Burnout.

Lưu ý:

Không loại bỏ các biến khác ở bước này.

Việc đánh giá tầm quan trọng của biến sẽ được thực hiện trong Feature Validation.

In [19]:
research_variables = [
    "burnout_risk",
    "emotional_exhaustion",
    "stress_level",
    "mental_fatigue",
    "daily_screen_time",
    "social_media_hours",
    "doomscrolling_duration",
    "late_night_device_usage",
    "notification_count",
    "smartphone_unlocks",
    "app_switch_frequency",
    "concentration_score",
    "distraction_frequency",
    "focus_sessions",
    "deep_work_hours",
    "task_completion_rate",
    "motivation_level",
    "sleep_hours",
    "sleep_quality",
    "productivity_score",
    "work_mode",
    "device_usage_type"
]

research_inventory = pd.DataFrame({
    "variable": research_variables,
    "available": [
        col in df_cleaned.columns
        for col in research_variables
    ]
})

display(research_inventory)

,variable,available
0,burnout_risk,True
1,emotional_exhaustion,True
2,stress_level,True
3,mental_fatigue,True
4,daily_screen_time,True
5,social_media_hours,True
6,doomscrolling_duration,True
7,late_night_device_usage,True
8,notification_count,True
9,smartphone_unlocks,True


## 8. Outlier Flagging

Phần này chỉ đánh dấu sự tồn tại của outlier bằng phương pháp IQR.

Không thực hiện loại bỏ outlier trong notebook này.

Các quyết định xử lý outlier sẽ được xem xét ở giai đoạn EDA và Feature Validation.

In [20]:
outlier_summary = []

for col in numerical_columns:

    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = (
        (df_cleaned[col] < lower)
        |
        (df_cleaned[col] > upper)
    ).sum()

    outlier_summary.append({
        "variable": col,
        "outlier_count": outliers
    })

outlier_summary_df = pd.DataFrame(
    outlier_summary
)

display(outlier_summary_df)

,variable,outlier_count
0,user_id,0
1,age,0
2,daily_screen_time,34754
3,social_media_hours,17325
4,doomscrolling_duration,16924
5,app_switch_frequency,0
6,notification_count,0
7,smartphone_unlocks,0
8,late_night_device_usage,0
9,focus_sessions,0


## 9. Final Data Quality Check

Kiểm tra chất lượng dữ liệu sau khi hoàn tất các bước tiền xử lý.

In [21]:
print("Final Shape:")
print(df_cleaned.shape)

print()

print("Remaining Missing Values:")
print(df_cleaned.isnull().sum().sum())

print()

print("Remaining Duplicates:")
print(df_cleaned.duplicated().sum())

Final Shape:
(5000000, 34)

Remaining Missing Values:
0

Remaining Duplicates:
0


## 10. Save Cleaned Dataset

Đây là output chính thức của notebook.

Dataset này sẽ được sử dụng trong:

- 03_EDA.ipynb
- 04_Feature_Validation_and_DBI.ipynb
- Cross-Dataset Comparative Analysis

In [22]:
cleaned_output_path = "../../data/processed/international_dataset/digital_burnout_cleaned.csv"

df_cleaned.to_csv(
    cleaned_output_path,
    index=False
)

print(
    f"Saved successfully: {cleaned_output_path}"
)

Saved successfully: ../../data/processed/international_dataset/digital_burnout_cleaned.csv
